# Pipeline Medallion — Bronze / Prata / Ouro

Pipeline completo de processamento de dados de devices seguindo a arquitetura **Medallion**:

| Camada | Bucket | Descricao |
|--------|--------|-----------|
| **Landing** | `s3a://landing/` | Dados brutos em JSON (origem) |
| **Bronze** | `s3a://bronze/` | Dados crus + metadados de ingestao (Delta) |
| **Prata** | `s3a://prata/` | Dados limpos, tipados e deduplicados (Delta) |
| **Ouro** | `s3a://ouro/` | Agregacoes de negocio prontas para consumo (Delta) |

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, current_timestamp, lit, trim, upper, lower,
    from_unixtime, count, desc
)
from pyspark.sql.types import TimestampType
from delta.tables import DeltaTable

spark = SparkSession \
    .builder \
    .appName("medallion-pipeline") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} conectado ao cluster")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/25 03:21:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.5 conectado ao cluster


---
## Camada Bronze

Ingestao dos dados brutos do **landing** com adicao de metadados de controle:
- `_ingestion_timestamp`: quando o dado foi ingerido
- `_source_format`: formato de origem do arquivo

Os dados sao gravados em **Delta Lake** no modo `append`, simulando chegada continua de dados.

In [2]:
# =============================================
# BRONZE: Ingestao crua + metadados
# =============================================

LANDING_PATH = "s3a://landing/*.json"
BRONZE_PATH = "s3a://bronze/device/raw"

df_landing = spark.read \
    .format("json") \
    .option("inferSchema", "true") \
    .json(LANDING_PATH)

print(f"Registros lidos do landing: {df_landing.count()}")

df_bronze = df_landing \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source_format", lit("json"))

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .save(BRONZE_PATH)

print(f"Bronze gravado em {BRONZE_PATH}")

2026-02-25T03:02:29,663 [Thread-3] WARN  org.apache.hadoop.metrics2.impl.MetricsConfig [] - Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


2026-02-25T03:02:46,852 [task-starvation-timer] WARN  org.apache.spark.scheduler.TaskSchedulerImpl [] - Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
2026-02-25T03:03:01,851 [task-starvation-timer] WARN  org.apache.spark.scheduler.TaskSchedulerImpl [] - Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
2026-02-25T03:03:16,850 [task-starvation-timer] WARN  org.apache.spark.scheduler.TaskSchedulerImpl [] - Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
2026-02-25T03:03:31,851 [task-starvation-timer] WARN  org.apache.spark.scheduler.TaskSchedulerImpl [] - Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


Registros lidos do landing: 200


Bronze gravado em s3a://bronze/device/raw


In [3]:
# Validacao Bronze
df_bronze_check = spark.read.format("delta").load(BRONZE_PATH)
print(f"Bronze — total de registros: {df_bronze_check.count()}")
df_bronze_check.printSchema()
df_bronze_check.show(5, truncate=False)

2026-02-25T03:04:35,077 [Thread-3] WARN  org.apache.spark.sql.catalyst.util.SparkStringUtils [] - Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Bronze — total de registros: 200
root
 |-- build_number: long (nullable = true)
 |-- dt_current_timestamp: long (nullable = true)
 |-- id: long (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- model: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- serial_number: string (nullable = true)
 |-- uid: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- version: long (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_format: string (nullable = true)

+------------+--------------------+----+------------+-----------------+-----------------+------------------------------+------------------------------------+-------+-------+--------------------------+--------------+
|build_number|dt_current_timestamp|id  |manufacturer|model            |platform         |serial_number                 |uid                                 |user_id|version|_ingestion_timestamp      |_source_format|
+------------+---------------

---
## Camada Prata (Silver)

Limpeza e padronizacao dos dados:
- Converter `dt_current_timestamp` (epoch ms) para tipo `timestamp`
- Remover duplicatas por `id`
- Filtrar registros com `id` ou `uid` nulos
- Padronizar `manufacturer` (UPPER) e `platform` (lower)
- Renomear colunas para nomes mais claros

In [4]:
# =============================================
# PRATA: Limpeza e padronizacao
# =============================================

PRATA_PATH = "s3a://prata/device"

df_bronze_raw = spark.read.format("delta").load(BRONZE_PATH)
bronze_count = df_bronze_raw.count()

df_prata = df_bronze_raw \
    .filter(col("id").isNotNull() & col("uid").isNotNull()) \
    .dropDuplicates(["id"]) \
    .withColumn("event_timestamp", (col("dt_current_timestamp") / 1000).cast(TimestampType())) \
    .withColumn("manufacturer", upper(trim(col("manufacturer")))) \
    .withColumn("platform", lower(trim(col("platform")))) \
    .select(
        col("id").alias("device_id"),
        col("uid"),
        col("user_id"),
        col("manufacturer"),
        col("model"),
        col("platform"),
        col("version"),
        col("build_number"),
        col("serial_number"),
        col("event_timestamp"),
        col("_ingestion_timestamp")
    )

df_prata.write \
    .format("delta") \
    .mode("overwrite") \
    .save(PRATA_PATH)

prata_count = df_prata.count()
print(f"Bronze: {bronze_count} registros")
print(f"Prata:  {prata_count} registros")
print(f"Removidos: {bronze_count - prata_count} (duplicatas + nulos)")

Bronze: 200 registros
Prata:  200 registros
Removidos: 0 (duplicatas + nulos)


In [5]:
# Validacao Prata
df_prata_check = spark.read.format("delta").load(PRATA_PATH)
print(f"Prata — total de registros: {df_prata_check.count()}")
df_prata_check.printSchema()
df_prata_check.show(5, truncate=False)

Prata — total de registros: 200
root
 |-- device_id: long (nullable = true)
 |-- uid: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- model: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- version: long (nullable = true)
 |-- build_number: long (nullable = true)
 |-- serial_number: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)

+---------+------------------------------------+-------+------------+-------------------+-----------+-------+------------+------------------------------+-----------------------+--------------------------+
|device_id|uid                                 |user_id|manufacturer|model              |platform   |version|build_number|serial_number                 |event_timestamp        |_ingestion_timestamp      |
+---------+------------------------------------+-------+------------+-------------------+--

---
## Camada Ouro (Gold)

Agregacoes de negocio prontas para consumo pelo Dremio / Metabase:
1. **Devices por fabricante** — ranking dos maiores fabricantes
2. **Devices por plataforma** — distribuicao por SO
3. **Resumo fabricante x plataforma** — tabela cruzada

In [6]:
# =============================================
# OURO: Agregacoes de negocio
# =============================================

OURO_FABRICANTE_PATH = "s3a://ouro/device/por_fabricante"
OURO_PLATAFORMA_PATH = "s3a://ouro/device/por_plataforma"
OURO_RESUMO_PATH = "s3a://ouro/device/fabricante_plataforma"

df_silver = spark.read.format("delta").load(PRATA_PATH)

# --- 1. Devices por fabricante ---
df_por_fabricante = df_silver \
    .groupBy("manufacturer") \
    .agg(count("*").alias("total_devices")) \
    .orderBy(desc("total_devices"))

df_por_fabricante.write \
    .format("delta") \
    .mode("overwrite") \
    .save(OURO_FABRICANTE_PATH)

print("Devices por fabricante:")
df_por_fabricante.show(truncate=False)

Devices por fabricante:
+------------+-------------+
|manufacturer|total_devices|
+------------+-------------+
|HUAWEI      |28           |
|HP          |25           |
|DELL        |25           |
|ASUS        |23           |
|ONEPLUS     |22           |
|ACER        |20           |
|XIAMOMI     |20           |
|APPLE       |19           |
|LENOVO      |18           |
+------------+-------------+



In [7]:
# --- 2. Devices por plataforma ---
df_por_plataforma = df_silver \
    .groupBy("platform") \
    .agg(count("*").alias("total_devices")) \
    .orderBy(desc("total_devices"))

df_por_plataforma.write \
    .format("delta") \
    .mode("overwrite") \
    .save(OURO_PLATAFORMA_PATH)

print("Devices por plataforma:")
df_por_plataforma.show(truncate=False)

Devices por plataforma:
+-----------------+-------------+
|platform         |total_devices|
+-----------------+-------------+
|android os       |20           |
|ios              |19           |
|windows phone    |17           |
|windows 10 mobile|16           |
|windows 8        |15           |
|windows 10       |15           |
|firefox os       |14           |
|windows rt       |13           |
|blackberry       |13           |
|danger os        |13           |
|android          |12           |
|ubuntu touch     |12           |
|windows 8.1      |11           |
|webos            |10           |
+-----------------+-------------+



In [8]:
# --- 3. Resumo fabricante x plataforma ---
df_resumo = df_silver \
    .groupBy("manufacturer", "platform") \
    .agg(count("*").alias("total_devices")) \
    .orderBy(desc("total_devices"))

df_resumo.write \
    .format("delta") \
    .mode("overwrite") \
    .save(OURO_RESUMO_PATH)

print("Resumo fabricante x plataforma (top 20):")
df_resumo.show(20, truncate=False)

Resumo fabricante x plataforma (top 20):
+------------+-----------------+-------------+
|manufacturer|platform         |total_devices|
+------------+-----------------+-------------+
|HP          |windows 8        |5            |
|ASUS        |windows phone    |4            |
|APPLE       |android os       |4            |
|DELL        |firefox os       |4            |
|XIAMOMI     |windows 10 mobile|4            |
|HP          |blackberry       |4            |
|HUAWEI      |windows rt       |4            |
|DELL        |android os       |4            |
|HUAWEI      |android os       |3            |
|ONEPLUS     |blackberry       |3            |
|ACER        |ios              |3            |
|ONEPLUS     |windows phone    |3            |
|HUAWEI      |windows phone    |3            |
|XIAMOMI     |android          |3            |
|ACER        |windows 10       |3            |
|APPLE       |firefox os       |3            |
|DELL        |danger os        |3            |
|ACER        |black

---
## Resumo do Pipeline

In [9]:
# Contagem final de cada camada
landing_count = spark.read.json("s3a://landing/*.json").count()
bronze_count = spark.read.format("delta").load(BRONZE_PATH).count()
prata_count = spark.read.format("delta").load(PRATA_PATH).count()
ouro_fab = spark.read.format("delta").load(OURO_FABRICANTE_PATH).count()
ouro_plat = spark.read.format("delta").load(OURO_PLATAFORMA_PATH).count()
ouro_resumo = spark.read.format("delta").load(OURO_RESUMO_PATH).count()

print("=" * 50)
print("RESUMO DO PIPELINE MEDALLION")
print("=" * 50)
print(f"Landing  (JSON):           {landing_count} registros")
print(f"Bronze   (Delta raw):      {bronze_count} registros")
print(f"Prata    (Delta limpo):    {prata_count} registros")
print(f"Ouro     - por fabricante: {ouro_fab} linhas")
print(f"Ouro     - por plataforma: {ouro_plat} linhas")
print(f"Ouro     - resumo cruzado: {ouro_resumo} linhas")
print("=" * 50)

RESUMO DO PIPELINE MEDALLION
Landing  (JSON):           200 registros
Bronze   (Delta raw):      200 registros
Prata    (Delta limpo):    200 registros
Ouro     - por fabricante: 9 linhas
Ouro     - por plataforma: 14 linhas
Ouro     - resumo cruzado: 104 linhas
2026-02-25T03:19:26,343 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl [] - Lost executor 0 on 172.18.0.9: worker lost: 172.18.0.9:34211 got disassociated
2026-02-25T03:19:26,349 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl [] - Lost executor 2 on 172.18.0.10: worker lost: 172.18.0.10:43969 got disassociated
2026-02-25T03:19:26,350 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl [] - Lost executor 1 on 172.18.0.11: worker lost: 172.18.0.11:46587 got disassociated
2026-02-25T03:19:26,351 [dispatcher-BlockManagerMaster] WARN  org.apache.spark.storage.BlockManagerMasterEndpoint [] - No more replicas available fo

2026-02-25T03:19:36,858 [dispatcher-event-loop-27] WARN  org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint [] - Connection to spark-master:7077 failed; waiting for master to reconnect...
2026-02-25T03:19:36,859 [dispatcher-event-loop-27] WARN  org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend [] - Disconnected from Spark cluster! Waiting for reconnection...
2026-02-25T03:19:36,859 [dispatcher-event-loop-27] WARN  org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint [] - Connection to spark-master:7077 failed; waiting for master to reconnect...
2026-02-25T03:19:49,870 [shutdown-hook-0] WARN  org.apache.spark.network.util.JavaUtils [] - Attempt to delete using native Unix OS command failed for path = /tmp/blockmgr-f7881c1c-08ca-45e3-88f7-0d9dab8ea566. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/blockmgr-f7881c1c-08ca-45e3-88f7-0d9dab8ea566
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(Java

In [ ]:
spark.stop()
print("SparkSession encerrada.")